# Notebook 10: 多算法对比 + 回测 -- 山东 15min 现货
## Multi-Agent Comparison + Backtest on Shandong 15-Minute Spot

**目标**: 训练 PPO/TD3/SAC 三种 RL 智能体，在历史数据上回测对比绩效
(PnL 曲线、Sharpe ratio、最大回撤)，理解算法特性与交易场景的匹配关系。

```
数据层 -> 预测层 -> 交易环境 -> RL 训练 -> 回测 -> 策略对比
  load     XGBoost     env(96d)   PPO/TD3/SAC   BacktestRunner.compare()
```

**数据源**: 山东 15min 现货，2024-07~2024-08 训练，2024-09 回测。

In [ ]:
# -- 导入 / Imports --
import os, json, warnings, logging
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)
pio.renderers.default = "notebook"

from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.features import FeatureEngineer
from ellectric.pipeline.forecaster import XGBoostForecaster
from ellectric.pipeline.trading_env import ElectricityMarketEnv, RewardRegistry
from ellectric.pipeline.rl_trainer import RLAgentFactory
from ellectric.pipeline.backtester import BacktestRunner, SUPPORTED_STRATEGIES
from ellectric.config import TimeConfig

print("All imports OK / 全部导入成功")
print(f"  TimeConfig.points_per_day = {TimeConfig.points_per_day}")
print(f"  Baseline strategies: {SUPPORTED_STRATEGIES}")

---
## 步骤 1: 加载数据 + 划分训练/回测 / Step 1: Load + Split Data

加载 3 个月数据，前两个月训练 XGBoost + RL，最后一个月回测。

In [ ]:
# -- 加载 + 清洗 + 划分 / Load + Clean + Split --
loader = create_loader("shandong")
df = loader.load_data("2024-07-01", "2024-09-30")
df = clean_data(df)

df_load = df[["timestamp", "load_mw"]].copy()
df_price = df[["timestamp", "load_mw", "da_price", "rt_price",
               "wind_actual_mw", "solar_actual_mw"]].copy()
df_price = df_price.rename(columns={"da_price": "price_da"})

TRAIN_END = "2024-09-01"
load_tr = df_load[df_load["timestamp"] < TRAIN_END].reset_index(drop=True)
price_tr = df_price[df_price["timestamp"] < TRAIN_END].reset_index(drop=True)
load_bt = df_load[df_load["timestamp"] >= TRAIN_END].reset_index(drop=True)
price_bt = df_price[df_price["timestamp"] >= TRAIN_END].reset_index(drop=True)

print(f"Train / 训练: {len(load_tr)} rows ({load_tr['timestamp'].min().date()} ~ {load_tr['timestamp'].max().date()})")
print(f"Backtest / 回测: {len(load_bt)} rows ({load_bt['timestamp'].min().date()} ~ {load_bt['timestamp'].max().date()})")
print(f"Load / 负荷: mean={df_load['load_mw'].mean():.0f} MW, max={df_load['load_mw'].max():.0f} MW")

---
## 步骤 2: 训练 XGBoost 预测器 + 创建环境 / Step 2: Train XGBoost + Create Env

In [ ]:
# -- 特征工程 + 训练 XGBoost --
fe = FeatureEngineer()
df_feat = fe.add_tier1_features(load_tr)
feature_cols = [c for c in df_feat.columns if c not in ("timestamp", "load_mw")]

lag = TimeConfig.points_per_day
X = df_feat[feature_cols].iloc[lag:]; y = df_feat["load_mw"].iloc[lag:]

MODEL_DIR = "../models"; os.makedirs(MODEL_DIR, exist_ok=True)
lf = XGBoostForecaster(n_estimators=100, max_depth=6, learning_rate=0.1)
r = lf.train_evaluate(X, y, n_splits=3, gap=lag)
lf.save_model(f"{MODEL_DIR}/xgboost_load.joblib")
print(f"XGBoost / 训练完成: MAE={r['metrics']['mae']:.1f} MW, RMSE={r['metrics']['rmse']:.1f} MW")

# -- 创建训练环境 / Create training env --
train_env = ElectricityMarketEnv(
    load_data=load_tr, price_data=price_tr, load_forecaster=lf,
    price_forecaster=None, initial_cash=0.0,
    max_capacity=load_tr["load_mw"].max(), reward_fn="profit_only")

print(f"Env ready / 环境就绪: action_dim={train_env.action_space.shape[0]}, "
      f"capacity={train_env._max_capacity:.0f} MW, ~{len(load_tr)//lag} episodes")

---
## 步骤 3: 训练三种 RL 算法 / Step 3: Train PPO, TD3, SAC

各 30,000 timesteps。若已训练过自动加载。
> 预计每个 5-10 分钟 (CPU)。可减少 TIMESTEPS 加速。

In [ ]:
TIMESTEPS = 30_000

def train_or_load(algo, env, force=False):
    path = f"{MODEL_DIR}/{algo}_trader_96d.zip"
    if os.path.exists(path) and not force:
        print(f"Loading / 加载 {algo.upper()} from {path}")
        return RLAgentFactory.load(algo, path, env), {"algo": algo, "loaded": True}
    print(f"Training / 训练 {algo.upper()} ({TIMESTEPS} steps)...")
    agent = RLAgentFactory.create(algo, env, tensorboard_log="../tb_logs", verbose=0)
    res = agent.train(total_timesteps=TIMESTEPS)
    agent.save(path)
    print(f"  Done. final_reward={res['final_reward']:.2f}, saved to {path}")
    return agent, {**res, "loaded": False}

# 训练三种算法
agent_ppo, info_ppo = train_or_load("ppo", train_env)
agent_td3, info_td3 = train_or_load("td3", train_env)
agent_sac, info_sac = train_or_load("sac", train_env)
print("\nAll agents ready / 所有智能体就绪")

---
## 步骤 4: 历史回测 / Step 4: Historical Backtest

在 2024-09 上回放各策略：oracle, persistence, mean, PPO, TD3, SAC。

In [ ]:
# -- 构建回测 runner / Build backtest runner --
def make_bt_env():
    return ElectricityMarketEnv(load_data=load_bt, price_data=price_bt,
        load_forecaster=lf, price_forecaster=None, reward_fn="profit_only",
        max_capacity=load_bt["load_mw"].max())

runner = BacktestRunner(make_bt_env)
bt_s = str(load_bt["timestamp"].min().date())
bt_e = str(load_bt["timestamp"].max().date())
print(f"Backtest period / 回测期间: {bt_s} ~ {bt_e}")

# -- 回放基线 + RL --
results = {}
for strat in ["baseline_persistence", "baseline_mean", "oracle"]:
    try:
        df = runner.replay(strat, load_bt, price_bt, bt_s, bt_e, strategy_name=strat)
        results[strat] = df
        print(f"  {strat}: P&L={df['pnl_cumulative'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  {strat}: FAIL - {e}")

for name, agent in [("PPO", agent_ppo), ("TD3", agent_td3), ("SAC", agent_sac)]:
    try:
        df = runner.replay(agent, load_bt, price_bt, bt_s, bt_e, strategy_name=name)
        results[name] = df
        print(f"  {name}: P&L={df['pnl_cumulative'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  {name}: FAIL - {e}")

print(f"\nAll done / 回放完成: {len(results)} strategies")

---
## 步骤 5: 策略对比 + 累计 P&L 图 / Step 5: Comparison + P&L Plot

In [ ]:
# -- 策略对比表 / Strategy Comparison --
if len(results) >= 2:
    comp = runner.compare(results)
    disp = comp.copy()
    for col in ["\u603b\u6536\u76ca", "\u6700\u5927\u56de\u64a4"]:
        if col in disp.columns:
            disp[col] = disp[col].apply(lambda x: f"{x:.2f}")
    if "\u590f\u666e\u6bd4\u7387" in disp.columns:
        disp["\u590f\u666e\u6bd4\u7387"] = disp["\u590f\u666e\u6bd4\u7387"].apply(lambda x: f"{x:.3f}")
    if "\u80dc\u7387" in disp.columns:
        disp["\u80dc\u7387"] = disp["\u80dc\u7387"].apply(lambda x: f"{x:.1%}")
    display(disp)
else:
    print("Not enough results / 回测结果不足")

# Oracle 相对性能
if "oracle" in results:
    oracle_pnl = results["oracle"]["pnl_hourly"].sum()
    print(f"Oracle P&L / Oracle 总收益: {oracle_pnl:.2f}")
    for name, df in results.items():
        if name == "oracle": continue
        pnl = df["pnl_hourly"].sum()
        ratio = (pnl / oracle_pnl * 100) if abs(oracle_pnl) > 1e-6 else 0
        print(f"  {name}: {pnl:.2f} ({ratio:.0f}% of oracle)")

# -- 累计 P&L 叠加图 / Cumulative P&L Overlay --
if len(results) >= 2:
    fig = runner.plot_comparison(results,
        title="Multi-Strategy Cumulative P&L / 多策略累计盈亏 — Shandong 15min Spot")
    fig.show()

---
## 讨论与思考 / Discussion & Questions

### 算法特性对比 / Algorithm Comparison
| 算法 | 特性 | 交易场景适配 |
|------|------|-------------|
| **PPO** | 收敛快、稳定、对超参数不敏感 | 快速验证和标准交易 |
| **TD3** | 处理噪声好、Q 值预估准 | 需要精确控制量的场景 |
| **SAC** | 探索性强、最大熵框架 | 非平稳市场 |

### 96 维 vs 24 维 / Impact of 96-dim
- 优势: 更精细的投标控制，更接近真实 15min 现货出清
- 代价: 更多 timesteps 才能收敛，维度灾难

### 思考题 / Questions
1. 哪种算法在这个交易任务中收敛最快？为什么？
2. 96 维动作空间对 TD3 (确定性策略) 和 SAC (随机策略) 的影响有何不同？
3. 真实市场中 oracle 可行吗？当前模型忽略了哪些真实约束？
4. 多智能体竞争环境中哪种算法可能胜出？

---
**下一步**: Notebook 11 -- SHAP 模型可解释性

In [ ]:
# -- 保存回测结果 / Save results --
try:
    comp.to_csv(f"{MODEL_DIR}/backtest_comparison_shandong.csv", index=False)
    print(f"Saved / 已保存: {MODEL_DIR}/backtest_comparison_shandong.csv")
except:
    pass
print("Notebook 10 complete / 完成!")